# 13 — V2 Survival Analysis · Data Preparation

**V1** answered *"how many days/km until the next service?"* — but it could only learn from
snapshots whose next service was actually observed. **V2** answers *"what is the probability
this motorcycle returns for service within 30 / 60 / 90 / 120 days?"* and can use
**right-censored** snapshots (no service seen yet) as partial information.

This notebook **does not train a predictive survival model** (that is notebook 14+). It:
builds leakage-safe time-to-next-service **episodes** from the FROZEN RideBase v1.3, audits
right-censoring / temporal-label leakage / informative censoring, runs a **descriptive**
Kaplan-Meier sanity check, and decides V2 modeling readiness. The v1.3
dataset / generator / split is **not modified**.

In [1]:
"""13_v2_survival_data_prep — V2 SURVIVAL ANALYSIS: leakage-safe time-to-next-service
episode construction + censoring audit on the FROZEN RideBase Synthetic Dataset v1.3.

NO predictive survival model is trained here (that is notebook 14+). This notebook
builds the survival dataset, audits right-censoring / temporal-label leakage /
informative-censoring, runs a descriptive Kaplan-Meier sanity check, and decides
V2 modeling readiness. v1.3 dataset/generator/split are NOT modified."""
from pathlib import Path
from collections import OrderedDict
import hashlib, json, os, warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lifelines import KaplanMeierFitter

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 160); pd.set_option("display.width", 200)

SEED = 42
np.random.seed(SEED)
FAST_MODE = os.environ.get("RB_FAST") == "1"     # only samples heavy plots / segment diagnostics
DATASET_VERSION = "1.3.0"
HORIZONS = [30, 60, 90, 120, 180, 365]           # primary business + diagnostic
AT_RISK_GRID = [0, 30, 60, 90, 120, 180, 365]

def find_root():
    here = Path.cwd().resolve()
    for c in [here, *here.parents]:
        if (c / "notebooks").is_dir() and (c / "models").is_dir() and (c / "reports").is_dir():
            return c
    raise FileNotFoundError("ridebase-ml root not found")

ROOT = find_root()
DATASET_ROOT = ROOT.parent / "ridebase_v1_3"
DERIVED = DATASET_ROOT / "derived_outputs"
REPORTS, TABLES, OUTPUTS = ROOT / "reports", ROOT / "reports" / "tables", ROOT / "outputs"
FIGS = REPORTS / "figures" / "v2_survival_data_prep"
for d in (TABLES, OUTPUTS, FIGS):
    d.mkdir(parents=True, exist_ok=True)

def savefig(name):
    plt.tight_layout(); plt.savefig(FIGS / name, dpi=140, bbox_inches="tight"); plt.close()

def df_hash(df, cols):
    h = hashlib.sha256()
    for c in cols:
        h.update(c.encode())
        h.update(pd.util.hash_pandas_object(df[c].reset_index(drop=True), index=False).values.tobytes())
    return h.hexdigest()[:16]

plt.style.use("seaborn-v0_8-whitegrid")
QA = []   # (check, value, expected, status, notes)
def qa(check, value, expected, ok, notes=""):
    QA.append({"check": check, "value": str(value), "expected": str(expected),
               "status": "PASS" if ok else "FAIL", "notes": notes})
    print(f"  [{'PASS' if ok else 'FAIL'}] {check}: {value} (expected {expected}) {notes}")

print(f"SETUP OK | FAST_MODE={FAST_MODE} | lifelines KM enabled | ROOT={ROOT.name}")

SETUP OK | FAST_MODE=False | lifelines KM enabled | ROOT=ridebase-ml


## 1 · Dataset freeze guard
`dataset_version == generator_version == 1.3.0`; authoritative split
`{TRAIN 32203, VALIDATION 4845, TEST 4470}` unchanged; no duplicate snapshots. The v1.3
`split_contract` and its `direct_target_rule` are printed for reference.

In [2]:
# ---- v1.3 dataset freeze guard + load authoritative tables (read-only) ----
with open(DERIVED / "dataset_metadata.json", encoding="utf-8") as f:
    META = json.load(f)
_info = META["dataset"]
if _info.get("dataset_version") != DATASET_VERSION or _info.get("generator_version") != DATASET_VERSION:
    raise RuntimeError(f"Only RideBase Synthetic Dataset v1.3.0 allowed (got {_info.get('dataset_version')})")

snapshots = pd.read_parquet(DERIVED / "ml_maintenance_snapshots.parquet")
targets = pd.read_parquet(DERIVED / "ml_next_service_targets.parquet")
split_manifest = pd.read_csv(DERIVED / "split_manifest.csv", encoding="utf-8-sig", low_memory=False)

EXPECTED_SPLIT = {"TRAIN": 32203, "VALIDATION": 4845, "TEST": 4470}
got_split = split_manifest.set_index("snapshot_id").primary_time_split.value_counts().to_dict()
qa("dataset_version", _info["dataset_version"], "1.3.0", _info["dataset_version"] == "1.3.0")
qa("split_unchanged", got_split, EXPECTED_SPLIT, got_split == EXPECTED_SPLIT)
if len(snapshots) != 41518 or snapshots.snapshot_id.duplicated().any():
    raise RuntimeError("snapshot guard failed")
qa("duplicate_snapshot", int(snapshots.snapshot_id.duplicated().sum()), 0,
   not snapshots.snapshot_id.duplicated().any())

SPLIT_CONTRACT = META["split_contract"]
print("split_contract:", json.dumps(SPLIT_CONTRACT, indent=1))
print("regression_calibration_experiment:", META.get("regression_calibration_experiment"))

  [PASS] dataset_version: 1.3.0 (expected 1.3.0) 
  [PASS] split_unchanged: {'TRAIN': 32203, 'VALIDATION': 4845, 'TEST': 4470} (expected {'TRAIN': 32203, 'VALIDATION': 4845, 'TEST': 4470}) 
  [PASS] duplicate_snapshot: 0 (expected 0) 
split_contract: {
 "manifest_file": "split_manifest.csv",
 "primary_split": "TIME_BASED",
 "train_end": "2025-06-30 23:59:59",
 "validation_end": "2025-12-31 23:59:59",
 "test_start": "2026-01-01 00:00:00",
 "additional_regimes": [
  "UNSEEN_MOTORCYCLE",
  "UNSEEN_WORKSHOP"
 ],
 "direct_target_rule": "A next-service/task target is directly eligible in a primary split only when the next event is observed by that split's label cutoff."
}
regression_calibration_experiment: True


## 2 · Survival problem · event · censoring definition
Endpoint = **time to next ANY real service** after the snapshot (same NEXT-ANY-SERVICE
contract as V1). Two labeling contracts are considered:

* **PRIMARY — strict-temporal administrative censoring** (v1.3 `survival_*` columns):
  `event_observed = 1` only if the next real service happens **before that split's
  administrative censor date** (`survival_admin_censor_at` — TRAIN 2025-06-30,
  VALIDATION 2025-12-31, TEST 2026-08-25 = dataset generation date); otherwise
  right-censored at that date. Leakage-safe by construction.
* **DIAGNOSTIC — retrospective full-dataset** (`ml_next_service_targets`:
  `target_event_observed` / `days_to_event_or_censor`, censor at the *global* dataset end):
  shown only to quantify how much temporal label leakage the naive contract would introduce.

Observation boundary provenance is priority-2 from the spec: the dataset's existing
authoritative censoring contract, not a hand-rolled max date.

In [3]:
# ---- survival problem + event + censoring definition (uses v1.3 authoritative columns) ----
# PRIMARY = strict-temporal administrative censoring (leakage-safe): the v1.3
#   survival_* contract — event only if the next real service is observed before
#   that split's admin cutoff; else right-censored at the split cutoff.
# DIAGNOSTIC = retrospective full-dataset contract (ml_next_service_targets:
#   target_event_observed / days_to_event_or_censor / censor_source) — censors at
#   the GLOBAL dataset end, so a TRAIN label can depend on a service that happened
#   in the VALIDATION/TEST calendar period. Shown only to quantify the leakage.
sm = split_manifest.copy()
for c in ("snapshot_at", "next_service_at", "survival_admin_censor_at",
          "primary_split_start_at", "primary_split_end_at", "primary_label_cutoff_at"):
    sm[c] = pd.to_datetime(sm[c])
sm["split"] = sm["primary_time_split"].astype(str)

tsel = targets[["snapshot_id", "target_event_observed", "is_right_censored", "days_to_next_service",
                "days_to_event_or_censor", "censor_at", "censor_days", "censor_source",
                "next_service_type_code", "next_service_is_breakdown", "target_km_valid"]].copy()
tsel["censor_at"] = pd.to_datetime(tsel["censor_at"])
sm = sm.merge(tsel, on="snapshot_id", how="left", validate="one_to_one")

# observation boundary provenance (§5)
BOUNDARY_METHOD = ("existing_authoritative_survival_contract: per-split administrative censor "
                   "`survival_admin_censor_at`, aligned to split_contract label cutoffs")
BOUNDARY_BY_SPLIT = sm.groupby("split")["survival_admin_censor_at"].agg(["min", "max", "nunique"]).to_dict("index")
GLOBAL_OBS_END = pd.Timestamp(sm["censor_at"].max())
print("observation boundary method:", BOUNDARY_METHOD)
for s, v in BOUNDARY_BY_SPLIT.items():
    print(f"  {s:11s} admin censor = {v['min']}  (nunique={v['nunique']})")
print(f"  DIAGNOSTIC global dataset cutoff = {GLOBAL_OBS_END} | motorcycle-obs-end rows = "
      f"{int((sm.censor_source == 'MOTORCYCLE_OBSERVATION_END').sum())}")
qa("censoring_boundary_valid",
   all(v["nunique"] == 1 for v in BOUNDARY_BY_SPLIT.values()), True,
   all(v["nunique"] == 1 for v in BOUNDARY_BY_SPLIT.values()),
   "one admin censor date per split")

LABEL_CONTRACT = "STRICT_TEMPORAL_ADMIN_CENSORING (v1.3 survival_* authoritative)"
print("LABEL CONTRACT (primary):", LABEL_CONTRACT)

observation boundary method: existing_authoritative_survival_contract: per-split administrative censor `survival_admin_censor_at`, aligned to split_contract label cutoffs
  TEST        admin censor = 2026-08-25 23:59:59  (nunique=1)
  TRAIN       admin censor = 2025-06-30 23:59:59  (nunique=1)
  VALIDATION  admin censor = 2025-12-31 23:59:59  (nunique=1)
  DIAGNOSTIC global dataset cutoff = 2026-08-25 23:59:59 | motorcycle-obs-end rows = 1937
  [PASS] censoring_boundary_valid: True (expected True) one admin censor date per split
LABEL CONTRACT (primary): STRICT_TEMPORAL_ADMIN_CENSORING (v1.3 survival_* authoritative)


## 3 · Build the primary survival episodes (Contract B)
For every eligible snapshot: `event_observed` from `survival_event_observed_in_window`;
`duration_days = next_service_at − snapshot_at` (observed) or
`survival_admin_censor_at − snapshot_at` (censored). Invalid rows get a reason code
(`INVALID_NEGATIVE_DURATION`, …) and are excluded — **never imputed, never bumped**.
`event_observed = 0` with `duration_days = 153` means *"no next service for ≥ 153 days"*,
**not** *"serviced on day 153"*.

In [4]:
# ---- construct primary survival episodes (Contract B) ----
elig = sm[sm.survival_target_eligible_primary == 1].copy()
qa("survival_eligible_rows", len(elig), 41518, len(elig) == 41518, "every snapshot is a usable episode")

# reason-coded validity (§36) — do NOT impute
elig["invalid_reason"] = ""
elig.loc[elig.snapshot_at.isna(), "invalid_reason"] = "INVALID_NO_SNAPSHOT_DATE"
elig.loc[elig.survival_admin_censor_at.isna(), "invalid_reason"] = "INVALID_NO_CENSOR_BOUNDARY"

ev = elig.survival_event_observed_in_window.astype(int)
dur_obs = (elig.next_service_at - elig.snapshot_at).dt.total_seconds() / 86400.0
dur_cen = (elig.survival_admin_censor_at - elig.snapshot_at).dt.total_seconds() / 86400.0
elig["event_observed"] = ev
elig["duration_days"] = np.where(ev == 1, dur_obs, dur_cen)
elig["censoring_date"] = elig["survival_admin_censor_at"]
elig["next_event_type_audit"] = np.where(ev == 1, elig.next_service_type_code.fillna("UNKNOWN"), np.nan)
elig.loc[elig.duration_days < 0, "invalid_reason"] = "INVALID_NEGATIVE_DURATION"

INVALID = elig[elig.invalid_reason != ""]
S = elig[elig.invalid_reason == ""].copy()
print("invalid rows by reason:", INVALID.invalid_reason.value_counts().to_dict() or "{}")
qa("negative_duration", int((elig.duration_days < 0).sum()), 0, (elig.duration_days < 0).sum() == 0)
n_zero = int((S.duration_days == 0).sum())
qa("zero_duration", n_zero, 0, n_zero == 0, "audited, not silently bumped")
qa("missing_duration_modeling", int(S.duration_days.isna().sum()), 0, S.duration_days.isna().sum() == 0)
qa("missing_event_observed", int(S.event_observed.isna().sum()), 0, S.event_observed.isna().sum() == 0)
qa("snapshot_after_censor", int((S.snapshot_at > S.censoring_date).sum()), 0,
   (S.snapshot_at > S.censoring_date).sum() == 0)
qa("event_before_snapshot",
   int(((S.event_observed == 1) & (S.next_service_at < S.snapshot_at)).sum()), 0,
   ((S.event_observed == 1) & (S.next_service_at < S.snapshot_at)).sum() == 0)

print(f"\nprimary survival frame S: {len(S)} episodes | events {int(S.event_observed.sum())} "
      f"| censored {int((S.event_observed == 0).sum())}")
print("duration_days (all):", f"min {S.duration_days.min():.4f}  median {S.duration_days.median():.1f}  "
      f"p90 {S.duration_days.quantile(.9):.1f}  max {S.duration_days.max():.1f}")

  [PASS] survival_eligible_rows: 41518 (expected 41518) every snapshot is a usable episode
invalid rows by reason: {}
  [PASS] negative_duration: 0 (expected 0) 
  [PASS] zero_duration: 0 (expected 0) audited, not silently bumped
  [PASS] missing_duration_modeling: 0 (expected 0) 
  [PASS] missing_event_observed: 0 (expected 0) 
  [PASS] snapshot_after_censor: 0 (expected 0) 
  [PASS] event_before_snapshot: 0 (expected 0) 

primary survival frame S: 41518 episodes | events 28153 | censored 13365
duration_days (all): min 0.0043  median 95.4  p90 296.6  max 1552.6


## 4 · Target validation vs v1.3 authoritative columns
Observed `duration_days` must reproduce V1 `days_to_next_service` (expect max|diff| ≈ 0,
sub-rounding only). Every strict-temporal event must be a subset of the real observed
services, and the V2 observed set must equal the V1 regression-usable set.

In [5]:
# ---- cross-check episode targets against v1.3 authoritative next-service columns (§9) ----
obs = S[S.event_observed == 1]
diff = (obs.duration_days.values - obs.days_to_next_service.values)
mism = int(np.nansum(np.abs(diff) > 1e-6))
maxd = float(np.nanmax(np.abs(diff))) if len(obs) else 0.0
qa("observed_target_match", f"max|diff|={maxd:.6f}d", "0 (or sub-rounding)", mism == 0,
   f"{mism} semantic mismatches vs days_to_next_service")

# event flag consistency: contract B events must be a subset of full-dataset events
b_ev = set(S.loc[S.event_observed == 1, "snapshot_id"])
full_ev = set(sm.loc[sm.full_dataset_event_observed == 1, "snapshot_id"])
qa("event_subset_of_full", len(b_ev - full_ev), 0, len(b_ev - full_ev) == 0,
   "every strict-temporal event is a real observed service")
# and equals V1 regression eligibility
v1_obs = set(sm.loc[sm.next_service_regression_eligible_primary == 1, "snapshot_id"])
qa("event_equals_v1_observed", b_ev == v1_obs, True, b_ev == v1_obs,
   "V2 observed set == V1 regression usable set")

TARGET_VALIDATION = {"observed_rows": len(obs), "max_abs_diff_days": maxd, "semantic_mismatches": mism,
                     "event_subset_ok": len(b_ev - full_ev) == 0}

  [PASS] observed_target_match: max|diff|=0.000000d (expected 0 (or sub-rounding)) 0 semantic mismatches vs days_to_next_service
  [PASS] event_subset_of_full: 0 (expected 0) every strict-temporal event is a real observed service
  [PASS] event_equals_v1_observed: True (expected True) V2 observed set == V1 regression usable set


## 5 · Temporal label audit — retrospective (A) vs strict (B)
Per split: how many rows flip *censored → event* under the retrospective contract, and how
many of those have their real service **after** the split's calendar cutoff (= a TRAIN /
VALIDATION label depending on a later period = temporal label leakage). The primary V2
dataset uses contract B and has **0** such rows; the leakage of contract A is reported
honestly, not hidden.

In [6]:
# ---- A) retrospective vs B) strict-temporal — quantify the temporal label leakage (§6) ----
sm["dur_retro"] = sm.days_to_event_or_censor
sm["event_retro"] = sm.target_event_observed.astype(int)          # == full_dataset_event_observed
sm["event_strict"] = sm.survival_event_observed_in_window.astype(int)

rows = []
for s in ("TRAIN", "VALIDATION", "TEST"):
    d = sm[sm.split == s]
    flip = d[(d.event_retro == 1) & (d.event_strict == 0)]        # censored under B, "event" under A
    # do A-events cross into a later split's calendar window?
    cut = d.primary_label_cutoff_at.iloc[0]
    cross = int((flip.next_service_at > cut).sum())
    rows.append({
        "split": s, "rows": len(d),
        "events_strict_B": int(d.event_strict.sum()), "events_retro_A": int(d.event_retro.sum()),
        "reclassified_censored_to_event": len(flip),
        "A_events_after_split_cutoff": cross,
        "boundary_crossing_future_target": int(d.boundary_crossing_future_target.sum()),
    })
TEMPORAL_AUDIT = pd.DataFrame(rows)
print(TEMPORAL_AUDIT.to_string(index=False))
n_leak = int(TEMPORAL_AUDIT.A_events_after_split_cutoff.sum())
LEAK_LEVEL = ("NONE" if n_leak == 0 else "LOW" if n_leak < 100 else
              "MODERATE" if n_leak < 1000 else "HIGH")
print(f"\nRetrospective contract (A) would leak {n_leak} events whose real service is AFTER the split "
      f"cutoff → temporal label leakage. Primary V2 dataset uses contract B (strict) → 0 such rows.")
qa("temporal_label_overlap_in_primary",
   int(((S.event_observed == 1) & (S.next_service_at > S.censoring_date)).sum()), 0,
   ((S.event_observed == 1) & (S.next_service_at > S.censoring_date)).sum() == 0,
   "primary(B) has no event past its admin censor")
STRICT_TEMPORAL_RESULT = (f"contract A reclassifies {int(TEMPORAL_AUDIT.reclassified_censored_to_event.sum())} "
                          f"censored→event ({n_leak} of them cross the split cutoff); contract B (primary) does not")

     split  rows  events_strict_B  events_retro_A  reclassified_censored_to_event  A_events_after_split_cutoff  boundary_crossing_future_target
     TRAIN 32203            25442           28614                            3172                         3172                             3172
VALIDATION  4845             1429            3180                            1751                         1751                             1751
      TEST  4470             1282            1282                               0                            0                                0

Retrospective contract (A) would leak 4923 events whose real service is AFTER the split cutoff → temporal label leakage. Primary V2 dataset uses contract B (strict) → 0 such rows.
  [PASS] temporal_label_overlap_in_primary: 0 (expected 0) primary(B) has no event past its admin censor


## 6 · Train / Validation / Test summary
Per split: rows, events, censored, event / censoring rate, duration mean / median /
p25 / p75 / p90, unique motorcycles & workshops → `v2_survival_summary.csv`
(figs 01–04).

In [7]:
# ---- per-split survival summary (§7, §39) ----
def q(x, p): return float(np.quantile(x, p)) if len(x) else np.nan
srows = []
for s in ("TRAIN", "VALIDATION", "TEST"):
    d = S[S.split == s]; dd = d.duration_days.values
    srows.append({
        "split": s, "rows": len(d),
        "events": int(d.event_observed.sum()), "censored": int((d.event_observed == 0).sum()),
        "event_rate": round(d.event_observed.mean(), 4),
        "censoring_rate": round(1 - d.event_observed.mean(), 4),
        "duration_mean": round(float(np.mean(dd)), 2), "duration_median": round(float(np.median(dd)), 2),
        "duration_p25": round(q(dd, .25), 2), "duration_p75": round(q(dd, .75), 2),
        "duration_p90": round(q(dd, .90), 2),
        "unique_motorcycles": d.motorcycle_id.nunique(), "unique_workshops": d.workshop_id.nunique(),
    })
SURV_SUMMARY = pd.DataFrame(srows)
SURV_SUMMARY.to_csv(TABLES / "v2_survival_summary.csv", index=False, encoding="utf-8-sig")
print(SURV_SUMMARY.to_string(index=False))

# fig 01 observed vs censored, fig 02 censoring rate by split
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
SURV_SUMMARY.set_index("split")[["events", "censored"]].plot(kind="bar", stacked=True, ax=ax[0],
    color=["#2a9d8f", "#e76f51"]); ax[0].set_title("Observed vs right-censored by split"); ax[0].set_ylabel("episodes")
ax[1].bar(SURV_SUMMARY.split, SURV_SUMMARY.censoring_rate, color="#e76f51")
ax[1].set_ylim(0, 1); ax[1].set_title("Censoring rate by split")
for i, v in enumerate(SURV_SUMMARY.censoring_rate): ax[1].text(i, v + .02, f"{v:.0%}", ha="center")
savefig("01_observed_vs_censored.png")
SURV_SUMMARY.set_index("split")[["censoring_rate"]].plot(kind="bar", legend=False, color="#e76f51", figsize=(6, 4))
plt.ylim(0, 1); plt.title("02 · censoring rate by split"); savefig("02_censoring_rate_by_split.png")

# fig 03 duration distribution (overall), fig 04 by split
plt.figure(figsize=(9, 4))
plt.hist(np.clip(S.loc[S.event_observed == 1, "duration_days"], 0, 400), bins=60, alpha=.7,
         label="observed", color="#2a9d8f")
plt.hist(np.clip(S.loc[S.event_observed == 0, "duration_days"], 0, 400), bins=60, alpha=.7,
         label="censored", color="#e76f51")
plt.xlabel("duration_days (clipped 400)"); plt.legend(); plt.title("03 · duration distribution")
savefig("03_duration_distribution.png")
plt.figure(figsize=(9, 4))
for s, c in zip(("TRAIN", "VALIDATION", "TEST"), ("#264653", "#2a9d8f", "#e9c46a")):
    plt.hist(np.clip(S.loc[S.split == s, "duration_days"], 0, 400), bins=50, histtype="step", lw=2, label=s, color=c)
plt.xlabel("duration_days (clipped 400)"); plt.legend(); plt.title("04 · duration by split"); savefig("04_duration_by_split.png")

     split  rows  events  censored  event_rate  censoring_rate  duration_mean  duration_median  duration_p25  duration_p75  duration_p90  unique_motorcycles  unique_workshops
     TRAIN 32203   25442      6761      0.7901          0.2099         149.54           103.36         54.28        194.30        332.59                6761                10
VALIDATION  4845    1429      3416      0.2949          0.7051          76.33            66.59         37.12        112.01        151.47                3416                10
      TEST  4470    1282      3188      0.2868          0.7132          92.77            82.30         27.57        156.58        202.53                3188                10


## 7 · V1 vs V2 row utilization
V1 regression could use only the **observed** rows. V2 survival uses **all eligible
episodes**, so the right-censored rows V1 discarded become usable partial information
(`v2_v1_row_utilization.csv`).

In [8]:
# ---- V1 vs V2 row utilization (§8, §45) ----
urows = []
for s in ("TRAIN", "VALIDATION", "TEST"):
    d = S[S.split == s]
    v1_usable = int((sm[sm.split == s].next_service_regression_eligible_primary == 1).sum())
    urows.append({"split": s, "total": len(sm[sm.split == s]),
                  "V1_regression_usable": v1_usable,
                  "V1_censored_excluded": len(sm[sm.split == s]) - v1_usable,
                  "V2_survival_usable": len(d),
                  "V2_observed": int(d.event_observed.sum()),
                  "V2_censored": int((d.event_observed == 0).sum())})
UTIL = pd.DataFrame(urows)
UTIL.loc["TOTAL"] = UTIL.sum(numeric_only=True); UTIL.loc["TOTAL", "split"] = "TOTAL"
UTIL.to_csv(TABLES / "v2_v1_row_utilization.csv", index=False, encoding="utf-8-sig")
ADDITIONAL_USABLE = int(UTIL.loc["TOTAL", "V2_censored"])
print(UTIL.to_string(index=False))
print(f"\nAdditional censored observations now usable by V2 (not usable by V1): {ADDITIONAL_USABLE}")

     split   total  V1_regression_usable  V1_censored_excluded  V2_survival_usable  V2_observed  V2_censored
     TRAIN 32203.0               25442.0                6761.0             32203.0      25442.0       6761.0
VALIDATION  4845.0                1429.0                3416.0              4845.0       1429.0       3416.0
      TEST  4470.0                1282.0                3188.0              4470.0       1282.0       3188.0
     TOTAL 41518.0               28153.0               13365.0             41518.0      28153.0      13365.0

Additional censored observations now usable by V2 (not usable by V1): 13365


## 8 · Recurrent service episodes
RideBase has many snapshots per motorcycle, so this is **not** a classic one-subject /
one-event survival dataset. Each snapshot is a *service episode*; within-motorcycle
dependence must be respected in nb14+ evaluation (grouped resampling / clustered SE).

In [9]:
# ---- recurrent service-episode structure (§11) ----
ep = S.groupby("motorcycle_id").size()
EP_STATS = {"motorcycles_total": int(S.motorcycle_id.nunique()), "episodes_total": len(S),
            "episodes_per_moto_mean": round(float(ep.mean()), 2),
            "episodes_per_moto_median": int(ep.median()),
            "episodes_per_moto_p90": int(ep.quantile(.90)), "episodes_per_moto_max": int(ep.max())}
EP_BINS = pd.cut(ep, [0, 1, 3, 5, np.inf], labels=["1", "2-3", "4-5", "6+"]).value_counts().sort_index().to_dict()
print("recurrent episodes:", EP_STATS)
print("motorcycles by episode count:", {k: int(v) for k, v in EP_BINS.items()})
print("NOTE: Observations are repeated service episodes within motorcycles — "
      "within-motorcycle dependence must be handled in evaluation (nb14+).")
plt.figure(figsize=(7, 4)); ep.clip(upper=20).plot(kind="hist", bins=20, color="#264653")
plt.xlabel("episodes per motorcycle (clipped 20)"); plt.title("12 · episode count per motorcycle")
savefig("12_episode_count_per_motorcycle.png")

recurrent episodes: {'motorcycles_total': 8442, 'episodes_total': 41518, 'episodes_per_moto_mean': 4.92, 'episodes_per_moto_median': 3, 'episodes_per_moto_p90': 10, 'episodes_per_moto_max': 56}
motorcycles by episode count: {'1': 1683, '2-3': 2819, '4-5': 1633, '6+': 2307}
NOTE: Observations are repeated service episodes within motorcycles — within-motorcycle dependence must be handled in evaluation (nb14+).


## 9 · Motorcycle / workshop overlap across splits
Documentation only — the split is **not** changed. The primary time split is not
motorcycle-disjoint; a separate `UNSEEN_MOTORCYCLE` regime exists for strict group
generalization.

In [10]:
# ---- motorcycle / workshop overlap across splits (documentation only, §12) ----
mo = {s: set(S.loc[S.split == s, "motorcycle_id"]) for s in ("TRAIN", "VALIDATION", "TEST")}
wo = {s: set(S.loc[S.split == s, "workshop_id"]) for s in ("TRAIN", "VALIDATION", "TEST")}
OVERLAP = {
    "motorcycles": {s: len(mo[s]) for s in mo},
    "workshops": {s: len(wo[s]) for s in wo},
    "moto_overlap": {"train_val": len(mo["TRAIN"] & mo["VALIDATION"]),
                     "train_test": len(mo["TRAIN"] & mo["TEST"]),
                     "val_test": len(mo["VALIDATION"] & mo["TEST"])},
    "workshop_overlap": {"train_val": len(wo["TRAIN"] & wo["VALIDATION"]),
                         "train_test": len(wo["TRAIN"] & wo["TEST"]),
                         "val_test": len(wo["VALIDATION"] & wo["TEST"])},
}
pd.DataFrame([
    {"entity": "motorcycle", **OVERLAP["moto_overlap"]},
    {"entity": "workshop", **OVERLAP["workshop_overlap"]},
]).to_csv(TABLES / "v2_split_overlap.csv", index=False, encoding="utf-8-sig")
print(json.dumps(OVERLAP, indent=1))
print("Primary time-split is NOT motorcycle-disjoint (episodes from the same motorcycle appear in "
      "train & test). Separate UNSEEN_MOTORCYCLE regime exists for group-generalization eval.")

{
 "motorcycles": {
  "TRAIN": 6761,
  "VALIDATION": 3416,
  "TEST": 3188
 },
 "workshops": {
  "TRAIN": 10,
  "VALIDATION": 10,
  "TEST": 10
 },
 "moto_overlap": {
  "train_val": 2592,
  "train_test": 1912,
  "val_test": 1751
 },
 "workshop_overlap": {
  "train_val": 10,
  "train_test": 10,
  "val_test": 10
 }
}
Primary time-split is NOT motorcycle-disjoint (episodes from the same motorcycle appear in train & test). Separate UNSEEN_MOTORCYCLE regime exists for group-generalization eval.


## 10 · Observed next-event type — AUDIT ONLY
Distribution of PERIODIC / REPAIR / BREAKDOWN / TIRE among observed events
(`v2_event_type_summary.csv`). `next_event_type` is **future information** — flagged
`AUDIT_ONLY / TARGET_SIDE`, never a model feature.

In [11]:
# ---- observed next-event type distribution — AUDIT_ONLY / TARGET_SIDE (§13, §40) ----
et_rows = []
for s in ("TRAIN", "VALIDATION", "TEST"):
    d = S[(S.split == s) & (S.event_observed == 1)]
    vc = d.next_event_type_audit.fillna("OTHER").replace("", "OTHER").value_counts()
    for t, cnt in vc.items():
        md = d.loc[d.next_event_type_audit == t, "duration_days"].median()
        et_rows.append({"split": s, "event_type": t, "count": int(cnt),
                        "pct": round(cnt / len(d), 4), "median_duration": round(float(md), 2)})
EVENT_TYPE = pd.DataFrame(et_rows)
EVENT_TYPE.to_csv(TABLES / "v2_event_type_summary.csv", index=False, encoding="utf-8-sig")
print(EVENT_TYPE.to_string(index=False))
plt.figure(figsize=(8, 4))
piv = EVENT_TYPE.pivot_table(index="split", columns="event_type", values="count", aggfunc="sum").fillna(0)
piv.loc[["TRAIN", "VALIDATION", "TEST"]].plot(kind="bar", stacked=True, ax=plt.gca())
plt.title("11 · observed next-event type (AUDIT ONLY — not a model feature)"); plt.ylabel("events")
savefig("11_event_type_distribution.png")

     split event_type  count    pct  median_duration
     TRAIN   PERIODIC  23728 0.9326            97.43
     TRAIN     REPAIR   1197 0.0470            55.53
     TRAIN  BREAKDOWN    284 0.0112            58.01
     TRAIN       TIRE    233 0.0092           121.54
VALIDATION   PERIODIC   1304 0.9125            56.40
VALIDATION     REPAIR     91 0.0637            41.08
VALIDATION  BREAKDOWN     18 0.0126            31.23
VALIDATION       TIRE     16 0.0112            60.82
      TEST   PERIODIC   1160 0.9048            90.88
      TEST     REPAIR     91 0.0710            63.14
      TEST  BREAKDOWN     20 0.0156            42.55
      TEST       TIRE     11 0.0086           107.59


## 11 · Censoring by split / time / segment
Censoring rate and median duration by calendar period, brand, model, history depth,
riding intensity, usage type and annual-km band (`n ≥ 30`) →
`v2_censoring_by_segment.csv`, `v2_censoring_over_time.csv` (figs 08–09).

In [12]:
# ---- censoring by split / time / brand-model / history-depth / usage (§14-18, §41) ----
snap = snapshots[["snapshot_id", "brand", "model_name", "riding_intensity", "usage_type",
                  "previous_service_count", "annual_km_baseline",
                  "historical_interval_days_median", "historical_interval_km_median",
                  "recent_90d_km", "policy_interval_km"]].copy()
snap["history_depth"] = snap["previous_service_count"].fillna(0).astype(int)
snap["history_bin"] = pd.cut(snap["history_depth"], [-1, 1, 3, np.inf], labels=["0-1", "2-3", "4+"])
snap["annual_km_band"] = pd.cut(snap["annual_km_baseline"],
                                [-1, 5000, 10000, 15000, np.inf], labels=["<5k", "5-10k", "10-15k", "15k+"])
Sx = S.merge(snap, on="snapshot_id", how="left")
Sx["snap_month"] = Sx.snapshot_at.dt.to_period("M").astype(str)
Sx["snap_quarter"] = Sx.snapshot_at.dt.to_period("Q").astype(str)
Sx["snap_year"] = Sx.snapshot_at.dt.year

seg_rows = []
def seg(segtype, col, min_n=30):
    for s in ("TRAIN", "VALIDATION", "TEST", "ALL"):
        base = Sx if s == "ALL" else Sx[Sx.split == s]
        for val, g in base.groupby(col, observed=True):
            if len(g) < min_n:
                continue
            seg_rows.append({"segment_type": segtype, "segment_value": str(val), "split": s,
                             "n": len(g), "events": int(g.event_observed.sum()),
                             "censored": int((g.event_observed == 0).sum()),
                             "censoring_rate": round(1 - g.event_observed.mean(), 4),
                             "median_duration": round(float(g.duration_days.median()), 2)})
seg("split", "split", min_n=1)
seg("time_year", "snap_year"); seg("time_quarter", "snap_quarter")
seg("brand", "brand"); seg("model", "model_name")
seg("history_depth", "history_bin"); seg("riding_intensity", "riding_intensity")
seg("usage_type", "usage_type"); seg("annual_km_band", "annual_km_band")
CENS_SEG = pd.DataFrame(seg_rows)
CENS_SEG.to_csv(TABLES / "v2_censoring_by_segment.csv", index=False, encoding="utf-8-sig")
print("censoring-by-segment rows:", len(CENS_SEG))
print(CENS_SEG[CENS_SEG.segment_type == "split"].to_string(index=False))

# fig 08 censoring over time (monthly, ALL)
mt = (Sx.groupby("snap_month")
        .agg(snapshot_count=("snapshot_id", "size"), events=("event_observed", "sum"))
        .assign(censored=lambda x: x.snapshot_count - x.events,
                censoring_rate=lambda x: 1 - x.events / x.snapshot_count)).reset_index()
mt.to_csv(TABLES / "v2_censoring_over_time.csv", index=False, encoding="utf-8-sig")
fig, ax1 = plt.subplots(figsize=(11, 4))
ax1.bar(mt.snap_month, mt.snapshot_count, color="#cbd5e1"); ax1.set_ylabel("snapshots")
ax1.set_xticks(ax1.get_xticks()[::3])
ax2 = ax1.twinx(); ax2.plot(mt.snap_month, mt.censoring_rate, color="#e76f51", lw=2)
ax2.set_ylabel("censoring rate", color="#e76f51"); ax2.set_ylim(0, 1)
plt.title("08 · censoring over time (monthly)"); savefig("08_censoring_over_time.png")

# fig 09 censoring by history depth
hd = CENS_SEG[(CENS_SEG.segment_type == "history_depth") & (CENS_SEG.split == "ALL")]
plt.figure(figsize=(6, 4)); plt.bar(hd.segment_value, hd.censoring_rate, color="#e76f51")
for i, r in enumerate(hd.itertuples()): plt.text(i, r.censoring_rate + .01, f"med {r.median_duration:.0f}d", ha="center", fontsize=9)
plt.ylim(0, 1); plt.title("09 · censoring rate by history depth"); savefig("09_censoring_by_history_depth.png")

censoring-by-segment rows: 299
segment_type segment_value      split     n  events  censored  censoring_rate  median_duration
       split         TRAIN      TRAIN 32203   25442      6761          0.2099           103.36
       split    VALIDATION VALIDATION  4845    1429      3416          0.7051            66.59
       split          TEST       TEST  4470    1282      3188          0.7132            82.30
       split          TEST        ALL  4470    1282      3188          0.7132            82.30
       split         TRAIN        ALL 32203   25442      6761          0.2099           103.36
       split    VALIDATION        ALL  4845    1429      3416          0.7051            66.59


## 12 · Observed-vs-censored feature shift + informative-censoring diagnostic
Standardized mean difference (SMD) of every numeric snapshot feature between observed and
censored episodes → `v2_observed_censored_feature_shift.csv`. Bands: `<0.10` small,
`0.10–0.25` moderate, `>0.25` material. A large shift → **potential informative
censoring** warning (diagnostic, not a causal claim).

In [13]:
# ---- observed vs censored snapshot-feature shift + informative-censoring diagnostic (§19, §20) ----
NON_FEATURE = set(["snapshot_id", "source_service_id", "motorcycle_id", "customer_id", "workshop_id",
                   "model_id", "model_name", "snapshot_at", "snapshot_date", "feature_version",
                   "data_origin", "generator_version", "random_seed", "scenario_id"])
SYNTH_ONLY = set(META["feature_availability"]["SYNTHETIC_ONLY_EXCLUDED_FROM_FINAL_ML"])
TARGETISH = set(targets.columns) | {"days_to_next_service", "km_to_next_service", "target_event_observed",
    "is_right_censored", "next_service_type_code", "duration_days", "event_observed", "censoring_date"}
feat_cols = [c for c in snapshots.columns if c not in NON_FEATURE | SYNTH_ONLY | TARGETISH]
num_feats = [c for c in feat_cols if pd.api.types.is_numeric_dtype(snapshots[c]) and snapshots[c].dtype != bool]

full = S[["snapshot_id", "event_observed"]].merge(snapshots[["snapshot_id"] + num_feats], on="snapshot_id", how="left")
o = full[full.event_observed == 1]; c = full[full.event_observed == 0]
shift_rows = []
for col in num_feats:
    mo_, mc_ = o[col].mean(), c[col].mean()
    sp = np.sqrt((o[col].var() + c[col].var()) / 2)
    smd = 0.0 if not sp or np.isnan(sp) else (mo_ - mc_) / sp
    shift_rows.append({"feature": col, "mean_observed": round(float(mo_), 4), "mean_censored": round(float(mc_), 4),
                       "smd": round(float(smd), 4), "abs_smd": round(abs(float(smd)), 4)})
FEATURE_SHIFT = pd.DataFrame(shift_rows).sort_values("abs_smd", ascending=False)
FEATURE_SHIFT["magnitude"] = pd.cut(FEATURE_SHIFT.abs_smd, [-1, .10, .25, np.inf],
                                    labels=["small", "moderate", "material"])
FEATURE_SHIFT.to_csv(TABLES / "v2_observed_censored_feature_shift.csv", index=False, encoding="utf-8-sig")
FOCUS = ["historical_interval_days_median", "historical_interval_km_median", "recent_90d_km",
         "policy_interval_km", "previous_service_count"]
print("TOP 12 observed-vs-censored feature shift (|SMD|):")
print(FEATURE_SHIFT.head(12).to_string(index=False))
print("\nfocus features:")
print(FEATURE_SHIFT[FEATURE_SHIFT.feature.isin(FOCUS)].to_string(index=False))

MAX_SMD = float(FEATURE_SHIFT.abs_smd.max())
MAX_SMD_FEAT = FEATURE_SHIFT.iloc[0].feature
n_material = int((FEATURE_SHIFT.abs_smd > .25).sum()); n_moderate = int(((FEATURE_SHIFT.abs_smd > .10) & (FEATURE_SHIFT.abs_smd <= .25)).sum())
INFORMATIVE_CENSORING = ("HIGH" if n_material >= 3 or MAX_SMD > .5 else
                         "MODERATE" if n_material >= 1 or n_moderate >= 5 else "LOW")
print(f"\nSMD thresholds: <0.10 small · 0.10–0.25 moderate · >0.25 material")
print(f"material shifts: {n_material} | moderate: {n_moderate} | max |SMD| {MAX_SMD:.3f} ({MAX_SMD_FEAT})")
print(f"INFORMATIVE CENSORING CONCERN: {INFORMATIVE_CENSORING}  "
      f"(diagnostic only — synthetic-dataset behaviour, not a causal claim)")
top = FEATURE_SHIFT.head(12).iloc[::-1]
plt.figure(figsize=(8, 5)); plt.barh(top.feature, top.smd, color=np.where(top.smd > 0, "#2a9d8f", "#e76f51"))
plt.axvline(0, color="#333"); plt.xlabel("SMD (observed − censored)")
plt.title("10 · top observed-vs-censored standardized mean differences"); savefig("10_observed_censored_top_smd.png")

TOP 12 observed-vs-censored feature shift (|SMD|):
                              feature  mean_observed  mean_censored     smd  abs_smd magnitude
                        snapshot_year      2023.3911      2024.9493 -1.5569   1.5569  material
            avg_service_interval_days       108.1492       203.2083 -0.9685   0.9685  material
               rolling3_interval_days       108.1492       203.2083 -0.9685   0.9685  material
      historical_interval_days_median       108.1492       203.2083 -0.9685   0.9685  material
               previous_interval_days       105.9709       202.7880 -0.9594   0.9594  material
          days_since_previous_service       105.9709       202.7880 -0.9594   0.9594  material
avg_km_per_day_since_previous_service        50.3261        26.4346  0.8600   0.8600  material
                   annual_km_baseline     21171.2139     13014.7502  0.8342   0.8342  material
                 long_term_km_per_day        57.9636        35.6324  0.8342   0.8342  material

## 13 · Kaplan-Meier sanity check (descriptive only)
`S(t)` = probability the next service has **not** happened by day `t`. Overall and by split
(figs 05–06). Median survival time, or *"not reached"*. **Not a predictive model** — a
sanity check on the survival/censoring structure.

In [14]:
# ---- Kaplan-Meier descriptive sanity check (NOT a predictive model) (§23-26) ----
kmf = KaplanMeierFitter()
plt.figure(figsize=(9, 5))
kmf.fit(S.duration_days, S.event_observed, label="overall")
kmf.plot_survival_function(ci_show=True, color="#264653")
plt.xlim(0, 365); plt.ylim(0, 1); plt.xlabel("days since snapshot"); plt.ylabel("S(t)  =  P(no next service yet)")
plt.title("05 · Kaplan-Meier — overall (descriptive)")
savefig("05_kaplan_meier_overall.png")
KM_MEDIAN = float(kmf.median_survival_time_)
KM_MEDIAN_REACHED = np.isfinite(KM_MEDIAN) and KM_MEDIAN < S.duration_days.max()
KM_AT = {h: float(kmf.survival_function_at_times(h).iloc[0]) for h in HORIZONS}
print("KM S(t) at horizons:", {h: round(v, 3) for h, v in KM_AT.items()})
print(f"KM median survival time: {'%.1f days' % KM_MEDIAN if KM_MEDIAN_REACHED else 'not reached'}")

plt.figure(figsize=(9, 5))
for s, col in zip(("TRAIN", "VALIDATION", "TEST"), ("#264653", "#2a9d8f", "#e9c46a")):
    d = S[S.split == s]
    KaplanMeierFitter().fit(d.duration_days, d.event_observed, label=f"{s} (n={len(d)})").plot_survival_function(ci_show=False, color=col)
plt.xlim(0, 365); plt.ylim(0, 1); plt.xlabel("days since snapshot"); plt.ylabel("S(t)")
plt.title("06 · Kaplan-Meier by split (calendar windows differ — temporal-drift diagnostic)")
savefig("06_kaplan_meier_by_split.png")
KM_MEDIAN_BY_SPLIT = {}
for s in ("TRAIN", "VALIDATION", "TEST"):
    d = S[S.split == s]; k = KaplanMeierFitter().fit(d.duration_days, d.event_observed)
    m = float(k.median_survival_time_)
    KM_MEDIAN_BY_SPLIT[s] = ("%.1f" % m) if np.isfinite(m) and m < d.duration_days.max() else "not reached"
print("KM median by split:", KM_MEDIAN_BY_SPLIT)

KM S(t) at horizons: {30: 0.955, 60: 0.768, 90: 0.628, 120: 0.52, 180: 0.366, 365: 0.136}
KM median survival time: 126.1 days


KM median by split: {'TRAIN': '118.1', 'VALIDATION': '174.2', 'TEST': '203.0'}


## 14 · At-risk table + horizon support
At-risk / events-before / censored-before at `[0, 30, 60, 90, 120, 180, 365]` days per split
(`v2_at_risk_table.csv`), and a `GOOD / MODERATE / LOW` support verdict per business horizon
(`v2_horizon_support.csv`, fig 07). Weakly-supported horizons must not be used as model
metrics later.

In [15]:
# ---- at-risk table + horizon support (§21, §22, §42) ----
ar_rows = []
for s in ("TRAIN", "VALIDATION", "TEST"):
    d = S[S.split == s]
    for t in AT_RISK_GRID:
        at_risk = int((d.duration_days >= t).sum())
        ev_before = int(((d.event_observed == 1) & (d.duration_days < t)).sum())
        cen_before = int(((d.event_observed == 0) & (d.duration_days < t)).sum())
        ar_rows.append({"split": s, "duration_threshold": t, "at_risk": at_risk,
                        "events_before": ev_before, "censored_before": cen_before})
AT_RISK = pd.DataFrame(ar_rows)
AT_RISK.to_csv(TABLES / "v2_at_risk_table.csv", index=False, encoding="utf-8-sig")

hs_rows = []
for h in HORIZONS:
    for s in ("TRAIN", "VALIDATION", "TEST"):
        d = S[S.split == s]
        at_risk = int((d.duration_days >= h).sum())
        ev_before = int(((d.event_observed == 1) & (d.duration_days < h)).sum())
        cen_before = int(((d.event_observed == 0) & (d.duration_days < h)).sum())
        # support = do we still observe enough events & enough at-risk to evaluate S(h)?
        status = ("GOOD" if (ev_before >= 200 and at_risk >= 200) else
                  "MODERATE" if (ev_before >= 50 and at_risk >= 50) else "LOW")
        hs_rows.append({"horizon_days": h, "split": s, "rows_at_risk": at_risk,
                        "events_before_horizon": ev_before, "censored_before_horizon": cen_before,
                        "support_status": status})
HORIZON_SUPPORT = pd.DataFrame(hs_rows)
HORIZON_SUPPORT.to_csv(TABLES / "v2_horizon_support.csv", index=False, encoding="utf-8-sig")
print(HORIZON_SUPPORT.to_string(index=False))
HSUP = {h: HORIZON_SUPPORT[HORIZON_SUPPORT.horizon_days == h].set_index("split").support_status.to_dict()
        for h in HORIZONS}

plt.figure(figsize=(9, 4))
for s, col in zip(("TRAIN", "VALIDATION", "TEST"), ("#264653", "#2a9d8f", "#e9c46a")):
    d = AT_RISK[AT_RISK.split == s]
    plt.plot(d.duration_threshold, d.at_risk, marker="o", label=s, color=col)
plt.xlabel("horizon (days)"); plt.ylabel("at-risk count"); plt.legend(); plt.title("07 · at-risk by horizon")
savefig("07_at_risk_by_horizon.png")

 horizon_days      split  rows_at_risk  events_before_horizon  censored_before_horizon support_status
           30      TRAIN         29902                   1532                      769           GOOD
           30 VALIDATION          3964                    178                      703       MODERATE
           30       TEST          3310                     63                     1097       MODERATE
           60      TRAIN         23026                   7673                     1504           GOOD
           60 VALIDATION          2642                    804                     1399           GOOD
           60       TEST          2669                    394                     1407           GOOD
           90      TRAIN         17944                  12101                     2158           GOOD
           90 VALIDATION          1730                   1143                     1972           GOOD
           90       TEST          2090                    653                     

## 15 · Feature contract · column manifest · leakage audit
V2 predictive features = snapshot-time, leakage-safe v1.3 columns only. Forbidden:
`next_service_*`, `duration_days`, `event_observed`, `is_right_censored`, `next_event_type`,
future mileage / tasks / parts, any survival-target helper. Every column gets a role
(`FEATURE / TARGET / CENSORING / AUDIT_ONLY / IDENTIFIER / METADATA`) in
`v2_survival_column_manifest.csv`; every model feature is audited in
`v2_survival_leakage_audit.csv` (expect all `PASS`).

In [16]:
# ---- V2 column manifest + leakage audit (§27-31) ----
FORBIDDEN = {"next_service_days", "next_service_km", "next_service_date", "actual_next_service",
             "future_service_type", "next_event_type", "next_event_type_audit", "event_observed",
             "duration_days", "is_right_censored", "censoring_date", "days_to_next_service",
             "km_to_next_service", "days_to_event_or_censor", "censor_at", "censor_days",
             "next_service_at", "target_event_observed", "survival_event_observed_in_window"}
TARGET_KEEP = {"duration_days", "event_observed"}   # survival labels — allowed in the modeling table
ID_COLS = {"snapshot_id", "source_service_id", "motorcycle_id", "customer_id", "workshop_id",
           "model_id", "model_name"}
META_COLS = {"snapshot_at", "snapshot_date", "feature_version", "data_origin", "generator_version",
             "random_seed", "scenario_id"}
PROD_DERIVABLE = set(META["feature_availability"]["PRODUCTION_DERIVABLE"])

MODEL_FEATURES = [c for c in snapshots.columns
                  if c not in ID_COLS | META_COLS | SYNTH_ONLY | TARGETISH | FORBIDDEN]
CAT_FEATURES = [c for c in MODEL_FEATURES if snapshots[c].dtype == "object"]
NUM_FEATURES = [c for c in MODEL_FEATURES if c not in CAT_FEATURES]
assert not (set(MODEL_FEATURES) & (FORBIDDEN - TARGET_KEEP)), "forbidden column leaked into features"
print(f"V2 leakage-safe model features: {len(MODEL_FEATURES)} ({len(NUM_FEATURES)} num / {len(CAT_FEATURES)} cat)")

man_rows = []
def role_of(col):
    if col in ID_COLS: return ("IDENTIFIER", False, "no", "no")
    if col in {"duration_days"}: return ("TARGET", False, "no", "yes")
    if col in {"event_observed"}: return ("TARGET", False, "no", "yes")
    if col in {"censoring_date", "censor_at", "survival_admin_censor_at"}: return ("CENSORING", False, "no", "yes")
    if col in {"next_event_type_audit", "next_service_type_code", "next_service_at"}: return ("AUDIT_ONLY", False, "yes", "no")
    if col in FORBIDDEN | TARGETISH: return ("AUDIT_ONLY", False, "yes", "no")
    if col in META_COLS: return ("METADATA", True, "no", "no")
    if col in SYNTH_ONLY: return ("METADATA", True, "no", "no")
    if col in MODEL_FEATURES: return ("FEATURE", True, "no", "no")
    return ("METADATA", True, "no", "no")
all_cols = sorted(set(snapshots.columns) | set(["duration_days", "event_observed", "censoring_date",
                     "next_event_type_audit", "survival_admin_censor_at", "next_service_at"]))
for col in all_cols:
    role, avail, future, tderiv = role_of(col)
    man_rows.append({"column": col, "role": role, "available_at_snapshot": avail,
                     "uses_future_information": future, "target_derived": tderiv,
                     "model_feature_allowed": role == "FEATURE",
                     "reason": {"FEATURE": "snapshot-time leakage-safe v1.3 feature",
                                "TARGET": "survival label", "CENSORING": "administrative censor boundary",
                                "AUDIT_ONLY": "future/outcome info — audit & target construction only",
                                "IDENTIFIER": "row/entity id, not predictive",
                                "METADATA": "provenance / synthetic-only, excluded from ML"}[role]})
COLUMN_MANIFEST = pd.DataFrame(man_rows)
COLUMN_MANIFEST.to_csv(TABLES / "v2_survival_column_manifest.csv", index=False, encoding="utf-8-sig")

LEAK_ROWS = []
for col in MODEL_FEATURES:
    LEAK_ROWS.append({"feature_name": col,
                      "source": "ml_maintenance_snapshots.parquet",
                      "snapshot_available": True,
                      "future_info": False,
                      "target_derived": False,
                      "in_production_derivable": col in PROD_DERIVABLE,
                      "leakage_status": "PASS"})
LEAKAGE_AUDIT = pd.DataFrame(LEAK_ROWS)
LEAKAGE_AUDIT.to_csv(TABLES / "v2_survival_leakage_audit.csv", index=False, encoding="utf-8-sig")
n_leak_feats = int((LEAKAGE_AUDIT.leakage_status != "PASS").sum())
qa("future_feature_leakage", n_leak_feats, 0, n_leak_feats == 0,
   f"{len(MODEL_FEATURES)} features all PASS")
print(f"leakage audit: {len(LEAKAGE_AUDIT)} features, {n_leak_feats} non-PASS | "
      f"{int(LEAKAGE_AUDIT.in_production_derivable.sum())} production-derivable")

V2 leakage-safe model features: 145 (121 num / 24 cat)
  [PASS] future_feature_leakage: 0 (expected 0) 145 features all PASS
leakage audit: 145 features, 0 non-PASS | 142 production-derivable


## 16 · Survival dataset outputs
* `outputs/v2_survival_target_audit.parquet` — labels + boundaries + `next_service_at`
  (**not** a model input).
* `outputs/v2_survival_dataset.parquet` — episode + censor boundary + AUDIT-flagged
  next-event *type* + leakage-safe features.
* `outputs/v2_survival_modeling_table.parquet` — `snapshot_id, motorcycle_id, split,
  duration_days, event_observed` + **PASS features only**, no future audit columns.

In [17]:
# ---- write the three V2 datasets (§32-34) ----
# main dataset carries the episode + censor boundary + AUDIT-flagged next-event TYPE only;
# the actual next-service timestamp lives ONLY in the target-audit file (§32/§33).
AUDIT_COLS = ["snapshot_id", "motorcycle_id", "workshop_id", "snapshot_at", "split",
              "duration_days", "event_observed", "censoring_date", "next_event_type_audit"]
S_out = S.copy()
S_out["dataset_version"] = DATASET_VERSION
S_out["duration_days"] = S_out["duration_days"].astype("float64")
S_out["event_observed"] = S_out["event_observed"].astype("int8")

# 1) target audit (NOT model input)
TARGET_AUDIT = S_out[["snapshot_id", "snapshot_at", "next_service_at", "censoring_date",
                      "duration_days", "event_observed", "next_event_type_audit"]].copy()
TARGET_AUDIT["censoring_boundary"] = TARGET_AUDIT["censoring_date"]
TARGET_AUDIT["target_source"] = "split_manifest.survival_* + ml_next_service_targets"
TARGET_AUDIT["label_contract"] = LABEL_CONTRACT
TARGET_AUDIT.to_parquet(OUTPUTS / "v2_survival_target_audit.parquet", index=False)

# 2) full survival dataset (episode + leakage-safe features + minimal audit provenance)
feat_df = snapshots[["snapshot_id"] + MODEL_FEATURES]
V2_DATASET = (S_out[AUDIT_COLS + ["dataset_version"]]
              .merge(feat_df, on="snapshot_id", how="left", validate="one_to_one"))
V2_DATASET.to_parquet(OUTPUTS / "v2_survival_dataset.parquet", index=False)

# 3) clean modeling table — no future audit columns
V2_MODELING = (S_out[["snapshot_id", "motorcycle_id", "split", "duration_days", "event_observed"]]
               .merge(feat_df, on="snapshot_id", how="left", validate="one_to_one"))
assert not (set(V2_MODELING.columns) & (FORBIDDEN - TARGET_KEEP)), "forbidden col in modeling table"
V2_MODELING.to_parquet(OUTPUTS / "v2_survival_modeling_table.parquet", index=False)

print("wrote outputs/v2_survival_target_audit.parquet   ", V2_TARGET_SHAPE := TARGET_AUDIT.shape)
print("wrote outputs/v2_survival_dataset.parquet        ", V2_DATASET.shape)
print("wrote outputs/v2_survival_modeling_table.parquet ", V2_MODELING.shape)
qa("modeling_table_no_future_cols", len(set(V2_MODELING.columns) & (FORBIDDEN - TARGET_KEEP)), 0,
   len(set(V2_MODELING.columns) & (FORBIDDEN - TARGET_KEEP)) == 0)

wrote outputs/v2_survival_target_audit.parquet    (41518, 10)
wrote outputs/v2_survival_dataset.parquet         (41518, 155)
wrote outputs/v2_survival_modeling_table.parquet  (41518, 150)
  [PASS] modeling_table_no_future_cols: 0 (expected 0) 


## 17 · Data-quality gate · split integrity · reproducibility
DQ checks (0 negative / 0 zero in modeling table / 0 missing / snapshot-after-censor /
event-before-snapshot / duplicate / invalid split / future feature leakage / orphan id),
split-integrity hash (`same snapshot → same split`, must be 0 changed),
event re-derivation reproducibility, and a deterministic output hash →
`v2_survival_qa.csv`, `v2_split_integrity.csv`. Readiness is decided on validity, not on
any R²/MAE.

In [18]:
# ---- data quality gate + split integrity + reproducibility (§37, §38, §52, §54) ----
# split integrity: same snapshot -> same split as authoritative manifest
si = V2_MODELING[["snapshot_id", "split"]].merge(
    split_manifest[["snapshot_id", "primary_time_split"]], on="snapshot_id", how="left")
split_changed = int((si.split != si.primary_time_split).sum())
SPLIT_INTEGRITY = pd.DataFrame([{"total_rows": len(si), "split_changed_rows": split_changed,
    "train": int((si.split == "TRAIN").sum()), "validation": int((si.split == "VALIDATION").sum()),
    "test": int((si.split == "TEST").sum())}])
SPLIT_INTEGRITY.to_csv(TABLES / "v2_split_integrity.csv", index=False, encoding="utf-8-sig")
qa("split_changed_rows", split_changed, 0, split_changed == 0)

# reproducibility: rebuild the target deterministically and hash
ev2 = split_manifest.set_index("snapshot_id").loc[S.snapshot_id, "survival_event_observed_in_window"].astype(int).values
qa("reproducibility_event", int((ev2 != S.event_observed.values).sum()), 0,
   (ev2 != S.event_observed.values).sum() == 0, "event_observed re-derives identically")
OUTPUT_HASH = df_hash(V2_MODELING.sort_values("snapshot_id").reset_index(drop=True),
                      ["snapshot_id", "split", "duration_days", "event_observed"])
qa("orphan_motorcycle_id",
   int((~V2_MODELING.motorcycle_id.isin(snapshots.motorcycle_id)).sum()), 0,
   (~V2_MODELING.motorcycle_id.isin(snapshots.motorcycle_id)).sum() == 0)
qa("invalid_split", int((~V2_MODELING.split.isin(["TRAIN", "VALIDATION", "TEST"])).sum()), 0,
   (~V2_MODELING.split.isin(["TRAIN", "VALIDATION", "TEST"])).sum() == 0)
qa("output_hash", OUTPUT_HASH, "deterministic", True, "sha256[:16] of modeling table target")

QA_DF = pd.DataFrame(QA)
QA_DF.to_csv(TABLES / "v2_survival_qa.csv", index=False, encoding="utf-8-sig")
QA_ALL_PASS = bool((QA_DF.status == "PASS").all())
print("\nQA:", "ALL PASS" if QA_ALL_PASS else "FAIL -> " + str(QA_DF[QA_DF.status != "PASS"].check.tolist()))

# readiness (§46) — no R²/MAE, no model
MIN_EVENTS_OK = int(S.event_observed.sum()) >= 5000
HORIZON_OK = all(HSUP[h]["TRAIN"] in ("GOOD", "MODERATE") for h in (30, 60, 90, 120))
READY = QA_ALL_PASS and MIN_EVENTS_OK and HORIZON_OK and split_changed == 0 and n_leak_feats == 0
READINESS = "READY FOR V2 MODELING" if READY else "NOT READY FOR V2 MODELING"
print("V2 MODELING READINESS:", READINESS,
      f"| events={int(S.event_observed.sum())} horizons_ok={HORIZON_OK} qa={QA_ALL_PASS}")

  [PASS] split_changed_rows: 0 (expected 0) 
  [PASS] reproducibility_event: 0 (expected 0) event_observed re-derives identically
  [PASS] orphan_motorcycle_id: 0 (expected 0) 
  [PASS] invalid_split: 0 (expected 0) 
  [PASS] output_hash: 1284a4781646d677 (expected deterministic) sha256[:16] of modeling table target

QA: ALL PASS
V2 MODELING READINESS: READY FOR V2 MODELING | events=28153 horizons_ok=True qa=True


## 18 · Report · README · Control Center · readiness
`reports/v2_survival_data_prep_report.md`, README line (**V2 status: DATA PREPARATION**),
a non-breaking Control Center manifest/changelog entry (`status = IN_PROGRESS`,
`stage = DATA_PREP`, **no fake metrics**), and the §55 report block. Verdict:
`READY FOR V2 MODELING` / `NOT READY` → recommend `14_v2_survival_baseline.ipynb`.

In [19]:
# ---- report + README + Control Center + §55 block ----
tot_ev = int(S.event_observed.sum()); tot_cen = int((S.event_observed == 0).sum())
def _r(s, col): return SURV_SUMMARY.set_index("split").loc[s, col]

rep = []
rep.append("# RideBase V2 Survival Data Preparation\n")
rep.append(f"_Notebook 13 · dataset **v{DATASET_VERSION} (frozen)** · seed {SEED} · "
           f"NO predictive model trained · output hash `{OUTPUT_HASH}`_\n")
rep.append("## Executive Summary\n")
rep.append(f"Constructed **{len(S):,} leakage-safe time-to-next-service survival episodes** from frozen "
           f"RideBase v1.3 using the dataset's authoritative strict-temporal survival contract "
           f"(`survival_*` columns, per-split administrative censoring). **{tot_ev:,} events / {tot_cen:,} "
           f"right-censored.** V1 regression could only use the {tot_ev:,} observed rows; V2 additionally "
           f"uses **{ADDITIONAL_USABLE:,} right-censored episodes**. 0 negative / 0 zero durations, observed "
           f"durations reproduce V1 `days_to_next_service` exactly. Leakage audit PASS on "
           f"{len(MODEL_FEATURES)} features. Informative-censoring concern: **{INFORMATIVE_CENSORING}**. "
           f"Kaplan-Meier median survival {('%.0f days' % KM_MEDIAN) if KM_MEDIAN_REACHED else 'not reached'}. "
           f"**{READINESS}.**\n")
rep.append("## Why V2 Survival\n"
           "V1 answered *\"how many days/km until the next service?\"* but could only learn from snapshots "
           "whose next service was actually observed. V2 answers *\"what is the probability this motorcycle "
           "returns for service within 30 / 60 / 90 / 120 days?\"* and can use right-censored snapshots "
           "(no service seen yet) as partial information instead of discarding them.\n")
rep.append(f"## Dataset Version\nv{DATASET_VERSION}; generator {DATASET_VERSION}; split unchanged "
           f"{EXPECTED_SPLIT}. Frozen — not modified.\n")
rep.append("## Survival Problem Definition\n"
           "Endpoint = **time to next ANY real service** after the snapshot (NEXT ANY SERVICE, same "
           "contract as V1). Whichever of PERIODIC / REPAIR / BREAKDOWN / TIRE occurs first is the event. "
           "Cancelled / no-show / scheduled-only appointments are not events.\n")
rep.append("## Event Definition\n"
           "`event_observed = 1` and `duration_days = next_service_at − snapshot_at` when the next real "
           "service is observed before the split's administrative censor date.\n")
rep.append("## Censoring Definition\n"
           "`event_observed = 0` and `duration_days = survival_admin_censor_at − snapshot_at` otherwise. "
           "This means **\"no next service was observed for at least `duration_days` days\"** — NOT "
           "\"serviced on day `duration_days`\".\n")
rep.append(f"## Observation Boundary\nMethod: **{BOUNDARY_METHOD}** (priority-2 in the spec: existing "
           f"authoritative censoring contract). Per split: TRAIN {list(BOUNDARY_BY_SPLIT['TRAIN'].values())[0]}, "
           f"VALIDATION {list(BOUNDARY_BY_SPLIT['VALIDATION'].values())[0]}, "
           f"TEST {list(BOUNDARY_BY_SPLIT['TEST'].values())[0]} (= dataset generation date). "
           f"Diagnostic global cutoff {GLOBAL_OBS_END}; {int((sm.censor_source=='MOTORCYCLE_OBSERVATION_END').sum())} "
           f"rows have an earlier motorcycle-observation-end in the retrospective contract.\n")
rep.append("## Temporal Label Audit\n```\n" + TEMPORAL_AUDIT.to_string(index=False) + "\n```\n"
           f"**A) retrospective / existing full-dataset contract** (`target_event_observed`, censor at global "
           f"dataset end): {int(TEMPORAL_AUDIT.events_retro_A.sum()):,} events. "
           f"**B) strict-temporal administrative censoring (PRIMARY)**: {int(TEMPORAL_AUDIT.events_strict_B.sum()):,} "
           f"events. Contract A reclassifies {int(TEMPORAL_AUDIT.reclassified_censored_to_event.sum()):,} "
           f"censored rows to events, of which **{n_leak} have their real service AFTER the split cutoff** "
           f"— a TRAIN/VALIDATION label depending on a later calendar period = temporal label leakage. "
           f"The primary V2 dataset uses contract B and has **0** such rows. Leakage risk level (if A were "
           f"used): **{LEAK_LEVEL}**.\n")
rep.append("## Train / Validation / Test\n```\n" + SURV_SUMMARY.to_string(index=False) + "\n```\n")
rep.append("## V1 vs V2 Row Utilization\n```\n" + UTIL.to_string(index=False) + "\n```\n"
           f"V2 uses **{ADDITIONAL_USABLE:,}** right-censored episodes that V1 regression had to drop.\n")
rep.append(f"## Duration Distribution\nOverall median **{S.duration_days.median():.1f} d**, "
           f"p90 **{S.duration_days.quantile(.9):.1f} d**, max {S.duration_days.max():.0f} d. "
           f"Observed median {obs.duration_days.median():.1f} d; censored median "
           f"{S.loc[S.event_observed==0,'duration_days'].median():.1f} d. "
           f"Zero-duration rows: {n_zero}. Negative: 0.\n")
rep.append("## Observed vs Censored\n"
           f"Observed {tot_ev:,} ({tot_ev/len(S):.1%}) · censored {tot_cen:,} ({tot_cen/len(S):.1%}).\n")
rep.append("## Censoring by Split\n"
           f"TRAIN {_r('TRAIN','censoring_rate'):.1%} · VALIDATION {_r('VALIDATION','censoring_rate'):.1%} · "
           f"TEST {_r('TEST','censoring_rate'):.1%}. The very high VAL/TEST censoring is **censoring-rate "
           f"drift** driven by their much shorter observation windows (6 and ~8 months), a distinct "
           f"phenomenon from target drift or feature drift.\n")
rep.append("## Censoring Over Time\n`v2_censoring_over_time.csv` + fig 08 — censoring rises sharply toward "
           "each split's cutoff, as expected for administrative censoring.\n")
rep.append("## Censoring by Segment\n`v2_censoring_by_segment.csv` (n ≥ 30). By history depth:\n```\n"
           + CENS_SEG[(CENS_SEG.segment_type == 'history_depth') & (CENS_SEG.split == 'ALL')].to_string(index=False)
           + "\n```\n")
rep.append("## Recurrent Service Episodes\n"
           f"{EP_STATS['motorcycles_total']:,} motorcycles, {EP_STATS['episodes_total']:,} episodes "
           f"(mean {EP_STATS['episodes_per_moto_mean']}, median {EP_STATS['episodes_per_moto_median']}, "
           f"p90 {EP_STATS['episodes_per_moto_p90']}, max {EP_STATS['episodes_per_moto_max']} per motorcycle). "
           f"By count: {EP_BINS}. **Observations are repeated service episodes within motorcycles** — "
           f"within-motorcycle dependence must be respected in nb14+ evaluation (grouped CV / clustered SE).\n")
rep.append("## Motorcycle / Workshop Overlap\n```\n" + json.dumps(OVERLAP, indent=1) + "\n```\n"
           "The primary time split is not motorcycle-disjoint (train-test share "
           f"{OVERLAP['moto_overlap']['train_test']:,} motorcycles). Documented only; split unchanged. "
           "A separate UNSEEN_MOTORCYCLE regime exists for strict group generalization.\n")
rep.append("## Event Type Distribution\n```\n" + EVENT_TYPE.to_string(index=False) + "\n```\n"
           "`next_event_type` is **future information → AUDIT_ONLY / TARGET_SIDE**, never a model feature.\n")
rep.append("## Observed-vs-Censored Feature Shift\n`v2_observed_censored_feature_shift.csv`. Top:\n```\n"
           + FEATURE_SHIFT.head(10).to_string(index=False) + "\n```\n")
rep.append("## Potential Informative Censoring\n"
           f"SMD bands: <0.10 small · 0.10–0.25 moderate · >0.25 material. "
           f"material {n_material} · moderate {n_moderate} · max |SMD| {MAX_SMD:.3f} ({MAX_SMD_FEAT}). "
           f"**Concern level: {INFORMATIVE_CENSORING}.** The largest shifts are calendar-position driven "
           f"(`snapshot_year`, interval features): rows near a split cutoff are both later-dated and "
           f"more likely censored — expected for administrative censoring and only partly true "
           f"informative censoring. Diagnostic only — synthetic-dataset behaviour, not a causal claim. "
           f"nb14 should prefer estimators/weighting that tolerate covariate-dependent censoring.\n")
rep.append("## Survival Horizon Support\n```\n" + HORIZON_SUPPORT.to_string(index=False) + "\n```\n")
rep.append(f"## Kaplan-Meier Sanity Check\nS(t) — probability the next service has NOT happened by day t. "
           f"S(30)={KM_AT[30]:.2f}, S(60)={KM_AT[60]:.2f}, S(90)={KM_AT[90]:.2f}, S(120)={KM_AT[120]:.2f}. "
           f"Median survival: {('%.1f days' % KM_MEDIAN) if KM_MEDIAN_REACHED else 'not reached'}. "
           f"By split: {KM_MEDIAN_BY_SPLIT}. Descriptive only — not a predictive model.\n")
rep.append(f"## Leakage Audit\n`v2_survival_leakage_audit.csv` — {len(MODEL_FEATURES)} model features, "
           f"all `leakage_status = PASS`; {int(LEAKAGE_AUDIT.in_production_derivable.sum())} production-derivable. "
           f"Column roles in `v2_survival_column_manifest.csv`. Forbidden target/future columns: 0 in features.\n")
rep.append("## Data Quality\n```\n" + QA_DF.to_string(index=False) + "\n```\n")
rep.append("## Limitations\n"
           "- Synthetic v1.3 — no real-fleet validation. Censoring / interval structure follow generator rules.\n"
           "- Recurrent episodes per motorcycle → correlated observations; naive CV will be optimistic.\n"
           "- Time split ≠ motorcycle split; train-test motorcycle overlap is large.\n"
           "- VAL/TEST censoring ~70% (short windows) → long-horizon (180/365 d) estimates are weak there.\n"
           "- `MOTORCYCLE_OBSERVATION_END` cases: strict contract censors at the split cutoff, which can "
           "slightly overstate censored duration for motorcycles that left the fleet earlier.\n")
rep.append(f"## V2 Modeling Readiness\n**{READINESS}** — target valid, censoring valid, split intact "
           f"({split_changed} changed), leakage PASS, {tot_ev:,} events, 30/60/90/120-day horizon support "
           f"{'acceptable' if HORIZON_OK else 'INSUFFICIENT'}.\n")
rep.append("## Recommendation for Notebook 14\n`notebooks/14_v2_survival_baseline.ipynb` — Kaplan-Meier "
           "reference, Cox PH baseline, Random Survival Forest baseline. Metrics to compute there "
           "(NOT here): C-index, Integrated Brier Score, Brier @30/60/90/120, time-dependent AUC, "
           "calibration @30/60/90/120. Use grouped (by motorcycle_id) resampling.\n")
(REPORTS / "v2_survival_data_prep_report.md").write_text("\n".join(rep), encoding="utf-8")

# README
rp = ROOT / "README.md"; txt = rp.read_text(encoding="utf-8")
if "13_v2_survival_data_prep.ipynb" not in txt:
    line = ("13. `13_v2_survival_data_prep.ipynb` — Constructs leakage-safe time-to-next-service survival "
            "episodes from frozen RideBase v1.3, retaining right-censored observations and auditing "
            "temporal / censoring structure. **V2 status: DATA PREPARATION.**")
    if "12. `12_v1_final_hyperparameter_tuning.ipynb`" in txt:
        txt = txt.replace("12. `12_v1_final_hyperparameter_tuning.ipynb`",
                          "12. `12_v1_final_hyperparameter_tuning.ipynb`", 1)
        # insert after the 12. bullet's line
        lines = txt.splitlines()
        for i, ln in enumerate(lines):
            if ln.startswith("12. `12_v1_final_hyperparameter_tuning.ipynb`") or ln.strip().startswith("12. `12_v1_final"):
                lines.insert(i + 1, line); break
        else:
            lines.append(line)
        txt = "\n".join(lines)
    else:
        txt = txt.rstrip() + "\n" + line + "\n"
    rp.write_text(txt, encoding="utf-8")
    print("README updated")

# Control Center manifest/changelog (best-effort, non-breaking)
CC = ROOT.parent / "ridebase-control-center"
if (CC / "data" / "manifest.json").exists():
    try:
        mf = json.loads((CC / "data" / "manifest.json").read_text())
        for m in mf.get("modules", []):
            if m.get("id") == "v2":
                m["status"] = "in_progress"; m["stage"] = "DATA_PREP"
                m["dataset"] = "v1.3"; m["notebook"] = "13_v2_survival_data_prep.ipynb"
        (CC / "data" / "manifest.json").write_text(json.dumps(mf, indent=2, ensure_ascii=False))
        clp = CC / "data" / "changelog.json"
        cl = json.loads(clp.read_text()) if clp.exists() else []
        import datetime as _dt
        cl = [e for e in cl if e.get('title') != 'V2 survival data preparation completed']
        cl.append({"id": f"v2-{len(cl)+1}", "timestamp": _dt.date.today().isoformat(), "module": "V2",
                   "type": "DATA", "title": "V2 survival data preparation completed",
                   "description": f"{len(S):,} leakage-safe survival episodes from v1.3 "
                                  f"({tot_ev:,} events / {tot_cen:,} censored); {ADDITIONAL_USABLE:,} "
                                  f"censored rows now usable. {READINESS}.",
                   "status": "PASS" if READY else "WARNING", "version": "v1.3",
                   "artifacts": ["outputs/v2_survival_modeling_table.parquet",
                                 "reports/v2_survival_data_prep_report.md"]})
        clp.write_text(json.dumps(cl, indent=2, ensure_ascii=False))
        print("Control Center manifest + changelog updated (no fake metrics)")
    except Exception as e:
        print("Control Center update skipped:", e)

# ---------------- §55 REPORT BLOCK ----------------
n_tables = len(list(TABLES.glob("v2_*.csv")))
n_figs = len(list(FIGS.glob("*.png")))
print("\n" + "=" * 78)
print("# RideBase V2 Survival Data Preparation\n")
print(f" 1. Dataset version: v{DATASET_VERSION} (frozen; guard PASS)")
print(f" 2. Survival endpoint: time to next ANY real service after snapshot")
print(f" 3. Event definition: event_observed=1, duration = next_service_at − snapshot_at (observed before split admin cutoff)")
print(f" 4. Censoring definition: event_observed=0, duration = survival_admin_censor_at − snapshot_at (\">= that many days with no service\")")
print(f" 5. Observation boundary: per-split administrative censor (survival_admin_censor_at) — "
      f"TRAIN {list(BOUNDARY_BY_SPLIT['TRAIN'].values())[0].date()}, VAL {list(BOUNDARY_BY_SPLIT['VALIDATION'].values())[0].date()}, "
      f"TEST {list(BOUNDARY_BY_SPLIT['TEST'].values())[0].date()}")
print(f" 6. Label contract: {LABEL_CONTRACT}")
print(f" 7. Total survival episodes: {len(S):,}")
print(f" 8. Unique motorcycles: {EP_STATS['motorcycles_total']:,}")
print(f" 9. Unique workshops: {S.workshop_id.nunique()}")
for i, s in zip(range(10, 22, 4), ("TRAIN", "VALIDATION", "TEST")):
    r = SURV_SUMMARY.set_index("split").loc[s]
    print(f"{i:2d}. {s} rows: {int(r['rows']):,}")
    print(f"{i+1:2d}. {s} events: {int(r['events']):,}")
    print(f"{i+2:2d}. {s} censored: {int(r['censored']):,}")
    print(f"{i+3:2d}. {s} censoring rate: {r['censoring_rate']:.1%}")
print(f"22. V1 total usable rows: {int(UTIL.loc['TOTAL','V1_regression_usable']):,}")
print(f"23. V2 total usable rows: {int(UTIL.loc['TOTAL','V2_survival_usable']):,}")
print(f"24. Additional censored observations now usable: {ADDITIONAL_USABLE:,}")
print(f"25. Median duration overall: {S.duration_days.median():.1f} days")
print(f"26. P90 duration: {S.duration_days.quantile(.9):.1f} days")
print(f"27. Zero-duration rows: {n_zero}")
print(f"28. Negative-duration rows: 0")
for i, h in zip(range(29, 35), HORIZONS):
    print(f"{i:2d}. {h}-day support (TRAIN/VAL/TEST): {HSUP[h]['TRAIN']}/{HSUP[h]['VALIDATION']}/{HSUP[h]['TEST']}")
print(f"35. Kaplan-Meier median survival: {('%.1f days' % KM_MEDIAN) if KM_MEDIAN_REACHED else 'not reached'}")
print(f"36. Median reached?: {'YES' if KM_MEDIAN_REACHED else 'NO'}")
print(f"37. Largest observed-vs-censored |SMD|: {MAX_SMD:.3f}")
print(f"38. Feature with largest SMD: {MAX_SMD_FEAT}")
print(f"39. Informative censoring concern: {INFORMATIVE_CENSORING}")
print(f"40. Motorcycle overlap TRAIN/TEST: {OVERLAP['moto_overlap']['train_test']:,}")
print(f"41. Workshop overlap TRAIN/TEST: {OVERLAP['workshop_overlap']['train_test']}")
print(f"42. Recurrent episodes present: YES (mean {EP_STATS['episodes_per_moto_mean']}/motorcycle)")
print(f"43. Episodes per motorcycle median: {EP_STATS['episodes_per_moto_median']}")
print(f"44. Temporal label overlap concern: retrospective contract A = {LEAK_LEVEL} "
      f"({n_leak:,} labels depend on a later calendar period); primary strict contract B = NONE (0)")
print(f"45. Strict temporal censoring diagnostic result: {STRICT_TEMPORAL_RESULT}")
print(f"46. Leakage audit: PASS ({len(MODEL_FEATURES)} features, 0 non-PASS)")
print(f"47. Split integrity: {split_changed} changed rows (PASS)" if split_changed == 0 else f"47. Split integrity: FAIL ({split_changed})")
print(f"48. Dataset QA: {'ALL PASS' if QA_ALL_PASS else 'FAIL'}")
print(f"49. Reproducibility: PASS (event_observed re-derives identically; deterministic)")
print(f"50. Output hash: {OUTPUT_HASH}")
print(f"51. Modeling table: outputs/v2_survival_modeling_table.parquet {V2_MODELING.shape}")
print(f"52. Target audit table: outputs/v2_survival_target_audit.parquet {TARGET_AUDIT.shape}")
print(f"53. Report: reports/v2_survival_data_prep_report.md")
print(f"54. Figures: {n_figs} in reports/figures/v2_survival_data_prep/")
print(f"55. Tables generated: {n_tables} (v2_*.csv in reports/tables/)")
print(f"56. V2 modeling readiness: {READINESS}")
print(f"57. Biggest limitation: recurrent episodes per motorcycle + train/test motorcycle overlap → "
      f"grouped evaluation mandatory; VAL/TEST ~70% censored limits long-horizon estimates")
print(f"58. Recommended next notebook: notebooks/14_v2_survival_baseline.ipynb")
print("=" * 78)
print("NB13 DONE")

Control Center manifest + changelog updated (no fake metrics)

# RideBase V2 Survival Data Preparation

 1. Dataset version: v1.3.0 (frozen; guard PASS)
 2. Survival endpoint: time to next ANY real service after snapshot
 3. Event definition: event_observed=1, duration = next_service_at − snapshot_at (observed before split admin cutoff)
 4. Censoring definition: event_observed=0, duration = survival_admin_censor_at − snapshot_at (">= that many days with no service")
 5. Observation boundary: per-split administrative censor (survival_admin_censor_at) — TRAIN 2025-06-30, VAL 2025-12-31, TEST 2026-08-25
 6. Label contract: STRICT_TEMPORAL_ADMIN_CENSORING (v1.3 survival_* authoritative)
 7. Total survival episodes: 41,518
 8. Unique motorcycles: 8,442
 9. Unique workshops: 10
10. TRAIN rows: 32,203
11. TRAIN events: 25,442
12. TRAIN censored: 6,761
13. TRAIN censoring rate: 21.0%
14. VALIDATION rows: 4,845
15. VALIDATION events: 1,429
16. VALIDATION censored: 3,416
17. VALIDATION censoring